# Valutazione e Ricalcolo V2 — con Emoji

Versione estesa di *Valutazione e Ricalcolo.ipynb* che include le **emoji** nel lessico ELIta.

**Differenza rispetto a V1**: la V1 filtrava le emoji dall'indice (`is_not_emoji`), trattandole come rumore.
Qui le includiamo: ELIta contiene 186 emoji annotate con vettori emotivi, e i commenti Reddit le usano come segnale espressivo (😂 → gioia, 😭 → tristezza, ecc.).

**Pipeline identica a V1**:
1. Calcolo della distintività per ogni entry (parole + emoji)
2. Selezione dei seed (top-50 più distinctive per emozione)
3. Costruzione dei centroidi come media dei seed
4. Similarità coseno tra ogni entry e i centroidi
5. Ricalcolo: `e* = α·cos + (1−α)·e`

## Import e caricamento dati

In [3]:
import pandas as pd
import numpy as np
import emoji
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Fase1.emotion_config import BASIC_EMOTIONS, EMOTION_COLORS

df_matrix = pd.read_csv("../Fase1/ELIta_INTENSITY_Matrix.csv", index_col=0)
df_matrix.index = df_matrix.index.astype(str)

# V2: nessun filtro emoji — includiamo tutto
df_emotions = df_matrix[BASIC_EMOTIONS].fillna(0)

def is_emoji_entry(text):
    return emoji.emoji_count(str(text)) > 0

emoji_mask = pd.Series(df_matrix.index).map(is_emoji_entry).values
n_emoji = int(emoji_mask.sum())
n_words = len(df_matrix) - n_emoji

print(f"Dataset completo ELIta : {len(df_matrix)} entry")
print(f"  Parole               : {n_words}")
print(f"  Emoji                : {n_emoji}")
print(f"\nV1 escludeva {n_emoji} emoji — V2 le include.")

Dataset completo ELIta : 6905 entry
  Parole               : 6719
  Emoji                : 186

V1 escludeva 186 emoji — V2 le include.


## Emoji in ELIta — statistiche

In [4]:
emoji_mask = df_matrix.index.map(is_emoji_entry)
df_emoji   = df_emotions[emoji_mask]

# Emozione dominante per ogni emoji
df_emoji_dom = df_emoji.copy()
df_emoji_dom['dominant'] = df_emoji[BASIC_EMOTIONS].idxmax(axis=1)
df_emoji_dom['max_score'] = df_emoji[BASIC_EMOTIONS].max(axis=1)

print(f"Emoji per emozione dominante:")
print(df_emoji_dom['dominant'].value_counts().to_string())
print()
print("Top 10 emoji per score massimo:")
display(df_emoji_dom.nlargest(10, 'max_score')[['dominant','max_score'] + BASIC_EMOTIONS].round(3))

Emoji per emozione dominante:
dominant
gioia          70
sorpresa       34
tristezza      28
aspettativa    15
rabbia         15
paura          12
disgusto        7
fiducia         5

Top 10 emoji per score massimo:


,dominant,max_score,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
parola,,,,,,,,,,
☹,tristezza,1.0,0.00,1.00,0.17,0.21,0.08,0.04,0.04,0.12
♥️,gioia,1.0,1.00,0.04,0.04,0.04,0.00,0.42,0.29,0.50
💞,gioia,1.0,1.00,0.04,0.04,0.04,0.00,0.92,0.54,0.71
💪,fiducia,1.0,0.46,0.12,0.00,0.04,0.00,1.00,0.25,0.88
😄,gioia,1.0,1.00,0.00,0.04,0.00,0.00,0.29,0.54,0.29
😘,gioia,1.0,1.00,0.00,0.04,0.00,0.04,0.75,0.54,0.42
😠,rabbia,1.0,0.00,0.25,1.00,0.25,0.42,0.00,0.00,0.00
😡,rabbia,1.0,0.12,0.25,1.00,0.21,0.42,0.04,0.33,0.17
🤞,aspettativa,1.0,0.29,0.00,0.00,0.12,0.00,0.33,0.17,1.00


## Calcolo della Distintività

Stessa formula di V1 — applicata ora anche alle emoji:

```
d = ((max1 − max2) / max1) × (max1 − mean)
```

In [5]:
def calculate_distinctiveness(row):
    sorted_values = sorted(row.values, reverse=True)
    max1 = sorted_values[0]
    max2 = sorted_values[1]
    mn   = np.mean(row.values)
    d_scores = {}
    for emo in BASIC_EMOTIONS:
        me = row[emo]
        if me == max1 and max1 > 0:
            d = ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)
            d_scores[emo] = d
        else:
            d_scores[emo] = 0
    return pd.Series(d_scores)

df_distinctiveness = df_emotions.apply(calculate_distinctiveness, axis=1)

print("Distintività calcolata su", len(df_distinctiveness), "entry (parole + emoji)")
df_distinctiveness

Distintività calcolata su 6905 entry (parole + emoji)


,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
parola,,,,,,,,
????,0.000000,0.000000,0.0,0.0,0.0,0.0,0.235348,0.000000
a_caso,0.000000,0.000000,0.0,0.0,0.0,0.0,0.192169,0.000000
a_malincuore,0.000000,0.468735,0.0,0.0,0.0,0.0,0.000000,0.000000
a_scanso_di,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.087931
abbagliante,0.000000,0.000000,0.0,0.0,0.0,0.0,0.054507,0.000000
...,...,...,...,...,...,...,...,...
🫤,0.000000,0.019310,0.0,0.0,0.0,0.0,0.000000,0.000000
🫥,0.000000,0.095179,0.0,0.0,0.0,0.0,0.000000,0.000000
🫦,0.000000,0.000000,0.0,0.0,0.0,0.0,0.060800,0.000000


## Emoji tra i seed — analisi

In [6]:
print("Emoji presenti tra le top-50 seed per ogni emozione:")
print("=" * 55)

emoji_seeds = {}
for emo in BASIC_EMOTIONS:
    top50 = df_distinctiveness[emo].sort_values(ascending=False).head(50).index
    emo_in_top50 = [w for w in top50 if is_emoji_entry(w)]
    emoji_seeds[emo] = emo_in_top50
    if emo_in_top50:
        print(f"  {emo:<15s}: {' '.join(emo_in_top50)}")
    else:
        print(f"  {emo:<15s}: (nessuna emoji tra i top-50)")

Emoji presenti tra le top-50 seed per ogni emozione:
  gioia          : 😆 🤣 😂 🤗 💃 🍌 ♥️
  tristezza      : ☹ 😔 😥 😭 🥀 🙁 😓 😪 🥲 🥺
  rabbia         : 😠 😡 👿 😤 🖕
  paura          : 💀 🥶 😱
  disgusto       : 🤢 🤮 🤧
  fiducia        : 🤝
  sorpresa       : 😵‍💫 🤯 😲 ⁉ 😮 💥 ❗ 🤨
  aspettativa    : 🤞 😏


## Costruzione dei centroidi

In [7]:
centroids = {}
for emo in BASIC_EMOTIONS:
    top_seeds = df_distinctiveness[emo].sort_values(ascending=False).head(50).index
    centroids[emo] = df_emotions.loc[top_seeds].mean().values

df_centroids = pd.DataFrame(centroids, index=BASIC_EMOTIONS).T
print("Centroidi V2 (parole + emoji):")
df_centroids.round(4)

Centroidi V2 (parole + emoji):


,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
gioia,0.8514,0.0486,0.0308,0.0394,0.0264,0.2100,0.1824,0.2198
tristezza,0.0296,0.8208,0.1950,0.2256,0.1138,0.0528,0.0746,0.1220
rabbia,0.0164,0.2424,0.8890,0.2798,0.2882,0.0390,0.1768,0.0986
paura,0.0460,0.2276,0.1798,0.7836,0.1158,0.0670,0.2020,0.1308
disgusto,0.0196,0.2216,0.2296,0.2082,0.8482,0.0202,0.1098,0.0644
fiducia,0.2510,0.0602,0.0454,0.0778,0.0204,0.7026,0.1154,0.3298
sorpresa,0.2294,0.1290,0.1596,0.2292,0.1120,0.1178,0.7922,0.3046
aspettativa,0.1628,0.0516,0.0644,0.1164,0.0358,0.2096,0.1468,0.7188


## Calcolo della Similarità Coseno

In [8]:
cos_sim_matrix = cosine_similarity(df_emotions.values, df_centroids.values)
df_cos_sim = pd.DataFrame(cos_sim_matrix, index=df_emotions.index, columns=BASIC_EMOTIONS)

print("Similarità coseno calcolata —", len(df_cos_sim), "entry × 8 centroidi")
print()

# Verifica sulle emoji: similarità coseno vs emozione dominante ELIta
print("Similarità coseno per alcune emoji chiave:")
test_emoji = ['😂','😭','😡','😱','🤢','💪','😲','😊']
for em in test_emoji:
    if em in df_cos_sim.index:
        dom_elita = df_emotions.loc[em, BASIC_EMOTIONS].idxmax()
        dom_cos   = df_cos_sim.loc[em, BASIC_EMOTIONS].idxmax()
        match = "✓" if dom_elita == dom_cos else "✗"
        print(f"  {em}  ELIta→{dom_elita:<15s}  cos→{dom_cos:<15s}  {match}")

Similarità coseno calcolata — 6905 entry × 8 centroidi

Similarità coseno per alcune emoji chiave:
  😂  ELIta→gioia            cos→gioia            ✓
  😭  ELIta→tristezza        cos→tristezza        ✓
  😡  ELIta→rabbia           cos→rabbia           ✓
  😱  ELIta→paura            cos→paura            ✓
  🤢  ELIta→disgusto         cos→disgusto         ✓
  💪  ELIta→fiducia          cos→fiducia          ✓
  😲  ELIta→sorpresa         cos→sorpresa         ✓
  😊  ELIta→gioia            cos→gioia            ✓


## Ricalcolo della Matrice V2

Formula: `e* = α·cos + (1−α)·e`

In [9]:
def recalculate_scores(df_orig, df_cos, alpha):
    return (alpha * df_cos) + ((1 - alpha) * df_orig)

df_alpha_02 = recalculate_scores(df_emotions, df_cos_sim, 0.2)
df_alpha_05 = recalculate_scores(df_emotions, df_cos_sim, 0.5)
df_alpha_08 = recalculate_scores(df_emotions, df_cos_sim, 0.8)

# Test su una parola e su un'emoji
for entry in ['amore', '😂', '😭']:
    if entry in df_emotions.index:
        print(f"\nEntry: '{entry}'")
        confronto = pd.DataFrame({
            'Originale'  : df_emotions.loc[entry],
            'Coseno'     : df_cos_sim.loc[entry],
            'α=0.5 V2'   : df_alpha_05.loc[entry],
        })
        print(confronto.round(3).to_string())


Entry: 'amore'
             Originale  Coseno  α=0.5 V2
gioia             1.00   0.782     0.891
tristezza         0.67   0.561     0.616
rabbia            0.42   0.461     0.440
paura             0.46   0.545     0.503
disgusto          0.04   0.293     0.167
fiducia           0.83   0.814     0.822
sorpresa          0.54   0.717     0.628
aspettativa       0.96   0.796     0.878

Entry: '😂'
             Originale  Coseno  α=0.5 V2
gioia             0.83   0.979     0.904
tristezza         0.00   0.098     0.049
rabbia            0.00   0.103     0.051
paura             0.00   0.172     0.086
disgusto          0.00   0.079     0.039
fiducia           0.12   0.520     0.320
sorpresa          0.33   0.606     0.468
aspettativa       0.21   0.490     0.350

Entry: '😭'
             Originale  Coseno  α=0.5 V2
gioia             0.00   0.118     0.059
tristezza         0.96   0.991     0.976
rabbia            0.17   0.481     0.326
paura             0.29   0.573     0.431
disgusto         

## Confronto V1 vs V2

Verifichiamo se includere le emoji nei seed ha modificato i centroidi e quindi i punteggi delle **parole** (non delle emoji).

In [14]:
# Carica la V1 per confronto
df_v1_05 = pd.read_csv("output_csv/elita_recalculated_0_5.csv", index_col=0)
df_v1_05.index = df_v1_05.index.astype(str)

# Maschera booleana numpy — evita Index.sum() non supportato
is_emoji_arr   = np.array([is_emoji_entry(w) for w in df_alpha_05.index])
words_only_idx = df_alpha_05.index[~is_emoji_arr]
common_words   = words_only_idx.intersection(df_v1_05.index)

df_v2_words = df_alpha_05.loc[common_words, BASIC_EMOTIONS]
df_v1_words = df_v1_05.loc[common_words, BASIC_EMOTIONS]

delta = (df_v2_words - df_v1_words).abs()

print(f"Parole comuni V1/V2: {len(common_words)}")
print()
print("Variazione media assoluta V1→V2 per emozione (α=0.5):")
print(delta.mean().round(6).to_string())
print()
print(f"Variazione massima assoluta: {delta.values.max():.6f}")
print()
print("Top 10 parole più cambiate (somma variazioni):")
top_changed = delta.sum(axis=1).nlargest(10)
display(pd.DataFrame({
    'delta_totale': top_changed,
    **{e: delta.loc[top_changed.index, e] for e in BASIC_EMOTIONS}
}).round(5))

Parole comuni V1/V2: 6719

Variazione media assoluta V1→V2 per emozione (α=0.5):
gioia          0.001911
tristezza      0.007389
rabbia         0.004966
paura          0.002753
disgusto       0.002364
fiducia        0.001825
sorpresa       0.007357
aspettativa    0.001207

Variazione massima assoluta: 0.026665

Top 10 parole più cambiate (somma variazioni):


,delta_totale,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
parola,,,,,,,,,
fermezza,0.04925,0.00277,0.02292,0.00128,0.00624,0.00390,0.00307,0.00632,0.00276
sergente,0.04658,0.00286,0.02301,0.00166,0.00339,0.00400,0.00342,0.00633,0.00192
tizio,0.04658,0.00286,0.02301,0.00166,0.00339,0.00400,0.00342,0.00633,0.00192
lotta,0.04645,0.00244,0.02006,0.00328,0.00578,0.00363,0.00318,0.00513,0.00295
petulanza,0.04639,0.00186,0.02362,0.00338,0.00655,0.00213,0.00294,0.00556,0.00035
dirigente,0.04637,0.00215,0.02197,0.00114,0.00351,0.00385,0.00151,0.00911,0.00313
riprovare,0.04636,0.00277,0.01871,0.00004,0.00674,0.00314,0.00224,0.01120,0.00153
insopportabile,0.04627,0.00171,0.02538,0.00542,0.00585,0.00097,0.00300,0.00391,0.00004
vago,0.04627,0.00153,0.02525,0.00614,0.00549,0.00080,0.00274,0.00406,0.00026


## Salvataggio output V2

In [13]:
from Fase2.support import save_alpha_csvs

output_dir = "output_csv"
saved_files = save_alpha_csvs(
    {0.2: df_alpha_02, 0.5: df_alpha_05, 0.8: df_alpha_08},
    output_dir=output_dir,
    base_name="elita_recalculated_v2"
)

print("Dataset V2 salvati:")
for f in saved_files:
    print(f"  {f}")

# np.array evita Index.sum() non supportato
is_emoji_arr = np.array([is_emoji_entry(w) for w in df_alpha_05.index])
n_emoji = int(is_emoji_arr.sum())
n_words = len(df_alpha_05) - n_emoji

print(f"\nEntry nel dataset V2 (α=0.5): {len(df_alpha_05)}")
print(f"  Parole: {n_words}")
print(f"  Emoji : {n_emoji}")

Dataset V2 salvati:
  output_csv/elita_recalculated_v2_0_2.csv
  output_csv/elita_recalculated_v2_0_5.csv
  output_csv/elita_recalculated_v2_0_8.csv

Entry nel dataset V2 (α=0.5): 6905
  Parole: 6719
  Emoji : 186
